#Beijing Multi-Site Air Quality
This data set includes hourly air pollutants data from 12 nationally-controlled air-quality monitoring sites. The air-quality data are from the Beijing Municipal Environmental Monitoring Center. The meteorological data in each air-quality site are matched with the nearest weather station from the China Meteorological Administration. The time period is from March 1st, 2013 to February 28th, 2017. Missing data are denoted as NA.

In [ ]:
%cd /content/drive/MyDrive/Colab/beijing-air-quality-data-analysis
!ls


/content/drive/MyDrive/Colab/beijing-air-quality-data-analysis
Beijing_air_quality_analysis.ipynb  Data  README.md


### Reading the dataset folder and combining the files into one dataset.







In [ ]:
import pandas as pd, numpy as np, os

DATA_DIR = "Data"
files = sorted((os.path.join(DATA_DIR, "PRSA_Data_*.csv")))
print(len(files), "files found",files)

df_raw = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
print(df_raw.shape)
df_raw.head()

12 files found ['Data/PRSA_Data_Aotizhongxin_20130301-20170228.csv', 'Data/PRSA_Data_Changping_20130301-20170228.csv', 'Data/PRSA_Data_Dingling_20130301-20170228.csv', 'Data/PRSA_Data_Dongsi_20130301-20170228.csv', 'Data/PRSA_Data_Guanyuan_20130301-20170228.csv', 'Data/PRSA_Data_Gucheng_20130301-20170228.csv', 'Data/PRSA_Data_Huairou_20130301-20170228.csv', 'Data/PRSA_Data_Nongzhanguan_20130301-20170228.csv', 'Data/PRSA_Data_Shunyi_20130301-20170228.csv', 'Data/PRSA_Data_Tiantan_20130301-20170228.csv', 'Data/PRSA_Data_Wanliu_20130301-20170228.csv', 'Data/PRSA_Data_Wanshouxigong_20130301-20170228.csv']
(420768, 18)


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


### Saving the combined dataset as combined_dataset.csv in our workfolder.

In [ ]:
# df_raw.to_csv('combined_dataset.csv',index=False)
df_combined = pd.read_csv('./combined_dataset.csv')

print(df_combined.shape)
df_combined.head()

(420768, 18)


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


### Checking missing values in all column, combining the year,month,day and hour column into one singular datetime column.


In [ ]:
df = df_combined.copy()
# keeping it for comparision


df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df = df.drop(columns="No").sort_values(["station", "datetime"]).reset_index(drop=True)

print("Duplicate station+time rows:", df.duplicated(["station", "datetime"]).sum())

miss_before = df.isna().sum().to_frame("missing")
miss_before["pct"] = (miss_before.missing / len(df) * 100).round(2)
miss_before[miss_before.missing > 0]


Duplicate station+time rows: 0


,missing,pct
PM2.5,8739,2.08
PM10,6449,1.53
SO2,9021,2.14
NO2,12116,2.88
CO,20701,4.92
O3,13277,3.16
TEMP,398,0.09
PRES,393,0.09
DEWP,403,0.10
RAIN,390,0.09


### Notes


*   PM2.5 and PM10 are types of microscopic particulate matter (dust, smoke, and liquid droplets) suspended in the air, defined by their diameter size in micrometers (μm). Readings of PM10 above 1000 are discarded as they are unrealistic under normal conditions.
*   for a gap of 6 hours or fewer, draw a straight line between the last real value before and the first real value after, and fill the gap along it.
*   whatever is still missing gets the typical value for that station, in that month, at that hour of the day.
*   interpolation runs first because it's more accurate, and the median is the fallback for what's left.
*   wd is text (NE, NNW, and so on), so you can't average or interpolate it, so we use forward filling(basically same logic as the rest), and use mode for larger gaps

In [23]:
pollutants = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
weather    = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
numeric    = pollutants + weather

# creating flags for the columns which will be imputed
df["imputed_any"]    = df[numeric + ["wd"]].isna().any(axis=1)
df["pm_inconsistent"] = df["PM2.5"] > df["PM10"]
df["extreme_pm25"]    = df["PM2.5"] > 1000

# short gaps of <6 hours are filled by interpolation, the data is already sorted by station
# and datetime so it is also time interpolation
df[numeric] = df.groupby("station")[numeric].transform(
    lambda s: s.interpolate(limit=6, limit_area="inside")
)

# fill the longer gaps by median of group of station,month and hour
grp = df.groupby(["station", "month", "hour"])
for c in numeric:
    df[c] = df[c].fillna(grp[c].transform("median"))

# Wind direction: similar logic but for text data, forward fill the short gaps and
# replace the longer gaps with mode
df["wd"] = df.groupby("station")["wd"].transform(lambda s: s.ffill(limit=6))
mode_wd = df.groupby(["station", "month"])["wd"].transform(
    lambda s: s.mode().iloc[0] if s.notna().any() else np.nan)
df["wd"] = df["wd"].fillna(mode_wd)


# 5) Before/after check
miss_after = df[numeric + ["wd"]].isna().sum()
print(pd.DataFrame({"before": miss_before.missing, "after": miss_after}).dropna())

       before  after
CO      20701    0.0
DEWP      403    0.0
NO2     12116    0.0
O3      13277    0.0
PM10     6449    0.0
PM2.5    8739    0.0
PRES      393    0.0
RAIN      390    0.0
SO2      9021    0.0
TEMP      398    0.0
WSPM      318    0.0
wd       1822    0.0


In [24]:
df.head()

,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,...,WSPM,station,datetime,imputed_any,pm_inconsistent,extreme_pm25,season,weekday,wd_deg,pm25_category
0,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,...,4.4,Aotizhongxin,2013-03-01 00:00:00,False,False,False,Spring,Friday,337.5,Excellent
1,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,...,4.7,Aotizhongxin,2013-03-01 01:00:00,False,False,False,Spring,Friday,0.0,Excellent
2,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,...,5.6,Aotizhongxin,2013-03-01 02:00:00,False,False,False,Spring,Friday,337.5,Excellent
3,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,...,3.1,Aotizhongxin,2013-03-01 03:00:00,False,False,False,Spring,Friday,315.0,Excellent
4,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,...,2.0,Aotizhongxin,2013-03-01 04:00:00,False,False,False,Spring,Friday,0.0,Excellent


In [ ]:

# derived these for future plans for dashboard
season_map = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
df["season"]  = df["month"].map(season_map)
df["weekday"] = df["datetime"].dt.day_name()
# df["wd_deg"]  = df["wd"].map({d: i * 22.5 for i, d in enumerate(
#     ["N","NNE","NE","ENE","E","ESE","SE","SSE","S","SSW","SW","WSW","W","WNW","NW","NNW"])})
df["pm25_category"] = pd.cut(df["PM2.5"], [0, 35, 75, 115, 150, 250, np.inf],
    labels=["Excellent", "Good", "Light", "Moderate", "Heavy", "Severe"])